## Some quick basics about pytorch

In [2]:
import torch 
import numpy as np 

### Tensors

In [3]:
x = torch.tensor([3, 4, 5])
y = torch.tensor([1, 2, 3])
x + y 

tensor([4, 6, 8])

In [4]:
x * y

tensor([ 3,  8, 15])

In [5]:
x = torch.tensor([[3, 4, 5], [3, 4, 5]])
y = torch.tensor([1, 2, 3])
x + y

tensor([[4, 6, 8],
        [4, 6, 8]])

In [6]:
x.size()

torch.Size([2, 3])

In [10]:
torch.matmul(x, x.T)

tensor([[50, 50],
        [50, 50]])

In [11]:
x.type()

'torch.LongTensor'

In [12]:
# Convert the types
z = x.double()
print(z.type())
print(x.type())

torch.DoubleTensor
torch.LongTensor


### Gradient 
Pytorch supports automatic differentiation, using the `backward()` method.

In [18]:
x = torch.tensor(1., requires_grad=True)  # Note that I created a floating point tensor
y = x * 2
print("Before backward(), grad is", x.grad)
y.backward()
print("After backward(), grad is", x.grad)

Before backward(), grad is None
After backward(), grad is tensor(2.)


#### The accumulation of gradient

In [19]:
# The gradient accumulates
# This is useful for increasing batch sizes
x = torch.tensor(1., requires_grad=True)  
y = x * 2
print("Before y.backward(), x.grad is", x.grad)
y.backward()
print("After y.backward(), x.grad is", x.grad)
z = x * 3
z.backward()
print("After z.backward(), x.grad is", x.grad)

Before y.backward(), x.grad is None
After y.backward(), x.grad is tensor(2.)
After z.backward(), x.grad is tensor(5.)


#### Gradients for leaf vs non-leaf nodes

In [20]:
# y is a "non-leaf node"
y.grad

/var/folders/5f/6nkzdmxs6hl7xl72dd7_m4cr0000gn/T/ipykernel_91566/1137013143.py:2: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


In [21]:
x = torch.tensor(1., requires_grad=True)
y = x * 2
y.retain_grad()  # Now we can see the gradient of a non-leaf node
y.backward()
y.grad

tensor(1.)

#### Backprop twice and retain_graph

In [23]:
x = torch.tensor(1., requires_grad=True)
y = x * 2
y.retain_grad()
y.backward(retain_graph=True)  # If you don't set retain_graph=True in the first run, the second run throws an error
print(x.grad, y.grad)
y.backward()
print(x.grad, y.grad)

tensor(2.) tensor(1.)
tensor(4.) tensor(2.)


### Module

- Initialization of the modules  
- Parameters vs Tensors


In [25]:
import torch.nn as nn 
from collections import OrderedDict 

class SimpleModule(nn.Module):
    def __init__(self, w):
        super().__init__()
        self.w = nn.Parameter(torch.tensor(w))

    def forward(self, x):
        return self.w * x

    def view_details(self, message=""):
        print(message)
        # You can visit the module parameters with the names, using this named_parameters() iterator
        for n,p in self.named_parameters():
            print("\n{}, {}, grad={}".format(n, p, p.grad))

def test_simple_module():
    m = SimpleModule(w=2.)
    y = m(3)  # Calling a module calls its `forward` method.
    m.view_details("After forward()")
    y.backward()
    m.view_details("After backward()")
    m.zero_grad()  # Alternatively, optimizers have implemented this method as well. Please refer to the next section
    m.view_details("After module.zero_grad()")

test_simple_module()

After forward()

w, Parameter containing:
tensor(2., requires_grad=True), grad=None
After backward()

w, Parameter containing:
tensor(2., requires_grad=True), grad=3.0
After module.zero_grad()

w, Parameter containing:
tensor(2., requires_grad=True), grad=None


### Optimizer

In [27]:
def optimize_simple_module():
    m = SimpleModule(w=2.)
    optimizer = torch.optim.Adam(
        m.parameters(),
        lr=3e-4
    )
    y = m(3)
    m.view_details("\nAfter forward()")
    optimizer.step()  # There's no gradient updates
    m.view_details("\nAfter optimizer.step() without backward")
    y.backward()
    optimizer.step()
    m.view_details("\nRetry optimizer.step() after backward()")
    optimizer.zero_grad()
    m.view_details("\nAfter optimizer.zero_grad()")

optimize_simple_module()


After forward()

w, Parameter containing:
tensor(2., requires_grad=True), grad=None

After optimizer.step() without backward

w, Parameter containing:
tensor(2., requires_grad=True), grad=None

Retry optimizer.step() after backward()

w, Parameter containing:
tensor(1.9997, requires_grad=True), grad=3.0

After optimizer.zero_grad()

w, Parameter containing:
tensor(1.9997, requires_grad=True), grad=None


### Practice question

**Multi-task learning with multiple loss functions**   
Here we want to build a simplified multi-task learning model. Here each data point contains three parts: a vector input with $D$ dimensions: $x\in \mathbb{R}^D$, and two scalar labels: $t_1$ and $t_2$. Prepare one batch of data -- these can be randomly generated. We are not letting the models learn some informative goals; instead, this part aims at reviewing and practicing the pytorch intricacies in the path that make the models work.  
 
Write a multi-task learning model in pytorch that contains three parameters:  
(1) An embedding layer, $E \in \mathbb{R}^{D} \rightarrow \mathbb{d}$ that projects the input into d-dimensional representations: $z = E(x)$  
(2) A layer, $A: \mathbb{R}^d \rightarrow \mathbb{R}$ that computes the score from $z$: $a = A(z)$.  
(3) A layer, $B: \mathbb{R}^d \rightarrow \mathbb{R}$ that computes the score from $z$: $b = B(z)$.  

There are two training objectives: (1) Minimize the mean-squared error between $a$ and $t_1$. (2) Minimize the mean-squared error between $b$ and $t_2$.  

Set up two optimizers. One for each training objective. These two optimizers should run for one step each.